# MedSegDiff — Sheffield Dataset — GPU Optimised (Aug 69 → 1)

Processes volumes **backwards from Aug_69** so two instances can split the workload
without coordination (run forward notebook alongside for Aug_1 → 34).

GPU optimisations vs `lambda_medsegdiff_sheffield.ipynb`:

| Change | Original | Optimised |
|---|---|---|
| Slice batching | batch_size=1 per DDIM call | INFER_BATCH (VRAM-scaled) |
| Precision | fp32 | fp16 autocast |
| Preprocessing | repeated per muscle | once per volume |

`INFER_BATCH` is auto-detected: UNet on 256² ≈ 150 MB peak VRAM per sample
(no-grad inference, fp16). On a 24 GB A10 this gives ~146; on an 80 GB H100 ~520 (capped at 512).

⚠️ **Checkpoints must exist** in `~/medsegdiff_ckpts/`.

Data: `~/sheffeld/20440164/Aug_N.dcm`
Output: `~/medsegdiff_sheffield_segs/Aug_N_seg.npz`

## 1 — Upload to Lambda
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/sheffeld \
  ubuntu@<YOUR-LAMBDA-IP>:~/

rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/medsegdiff_ckpts/ \
  ubuntu@<YOUR-LAMBDA-IP>:~/medsegdiff_ckpts/

rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/medsegdiff/ \
  ubuntu@<YOUR-LAMBDA-IP>:~/medsegdiff/
```

## 2 — Download results when done
```bash
rsync -avz --mkpath -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@<YOUR-LAMBDA-IP>:~/medsegdiff_sheffield_segs/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/medsegdiff/sheffield_segs/
```
**Terminate the instance when done.**

In [ ]:
import subprocess, sys

def _ensure(*pkgs):
    import importlib
    missing = [p for p in pkgs
               if importlib.util.find_spec(p.replace('-', '_')) is None]
    if missing:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(missing))
    else:
        print('Already installed:', ', '.join(pkgs))

_ensure('SimpleITK', 'tqdm', 'torchvision', 'pydicom')

import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
import glob, os, re, sys
import numpy as np
import torch
import torch.nn.functional as F
import pydicom

MEDSEGDIFF_DIR = os.path.expanduser('~/medsegdiff')
CKPT_DIR       = os.path.expanduser('~/medsegdiff_ckpts')
IMG_DIR        = os.path.expanduser('~/sheffeld/20440164')
OUTPUT_DIR     = os.path.expanduser('~/medsegdiff_sheffield_segs')

for path, label in [
    (MEDSEGDIFF_DIR, 'medsegdiff package'),
    (CKPT_DIR,       'checkpoints'),
    (IMG_DIR,        'Sheffield images'),
]:
    ok = os.path.isdir(path)
    print(f'{"OK" if ok else "MISSING"}: {label}')
    if not ok:
        raise FileNotFoundError(f'Upload {label} first')

os.makedirs(OUTPUT_DIR, exist_ok=True)

if MEDSEGDIFF_DIR not in sys.path:
    sys.path.insert(0, MEDSEGDIFF_DIR)

from dataset   import GT_LABELS, _norm
from unet      import UNet
from diffusion import GaussianDiffusion

IMG_SIZE   = 256
BASE_CH    = 64
T_DIM      = 256
T_STEPS    = 1000
DDIM_STEPS = 50
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MUSCLES    = list(GT_LABELS)

# ── GPU batch size ────────────────────────────────────────────────────────────
def _auto_infer_batch(device):
    """With flash attention the bottleneck is conv activations: ~100 MB per sample."""
    if str(device) == 'cpu':
        return 4
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    return max(8, min(256, int((vram_gb - 6) / 0.1)))

INFER_BATCH = _auto_infer_batch(DEVICE)

# Aug_69 → Aug_1, reverse order
dcm_files = sorted(
    [f for f in glob.glob(os.path.join(IMG_DIR, 'Aug_*.dcm'))
     if '_segmentations' not in f],
    key=lambda p: int(re.search(r'Aug_(\d+)\.dcm', p).group(1)),
    reverse=True,
)
print(f'Muscles     : {MUSCLES}')
print(f'Volumes     : {len(dcm_files)}  (Aug_69 → Aug_1)')
print(f'Device      : {DEVICE}')
print(f'INFER_BATCH : {INFER_BATCH}  (slices per DDIM forward pass)')

In [ ]:
models    = {}
diffusion = GaussianDiffusion(T=T_STEPS, device=DEVICE)

for muscle in MUSCLES:
    best_ckpt = os.path.join(CKPT_DIR, f'{muscle}_best.pt')
    if not os.path.exists(best_ckpt):
        print(f'[WARN] checkpoint not found: {best_ckpt}')
        continue
    ckpt       = torch.load(best_ckpt, map_location=DEVICE)
    saved_args = ckpt.get('args', {})
    img_ch     = saved_args.get('img_ch', 2)
    model = UNet(
        img_ch=img_ch,
        base=saved_args.get('base_ch', BASE_CH),
        t_dim=saved_args.get('t_dim', T_DIM),
    ).to(DEVICE)
    model.load_state_dict(ckpt['model'])
    model.eval()
    models[muscle] = (model, img_ch)
    print(f'  {muscle}: epoch {ckpt.get("epoch","?")}, '
          f'best Dice {ckpt.get("best_dice",0):.4f}, img_ch={img_ch}')

print(f'Loaded {len(models)}/{len(MUSCLES)} models.')

In [ ]:
# ── Flash-attention monkey-patch ──────────────────────────────────────────────
# The Attention class uses explicit O(N²) einsum (N=4096 at enc2 level).
# Replacing with F.scaled_dot_product_attention gives O(N) memory (flash attn)
# and removes the 17+ GB allocation that causes OOM at batch_size > ~40.
from unet import Attention

def _flash_forward(self, x: torch.Tensor) -> torch.Tensor:
    batch, ch, ht, wd = x.shape
    feat = self.norm(x)   # GroupNorm outputs fp32 even inside autocast
    qkv  = self.qkv(feat).reshape(batch, 3, self.heads, ch // self.heads, ht * wd)
    q, k, v = qkv.unbind(dim=1)          # (B, heads, d, N) — may be fp32
    q = q.transpose(-1, -2).half()        # (B, heads, N, d) fp16 → flash backend
    k = k.transpose(-1, -2).half()
    v = v.transpose(-1, -2).half()
    # SDPA uses flash attention only when dtype is fp16/bf16 — O(N) memory
    out = F.scaled_dot_product_attention(q, k, v)   # (B, heads, N, d) fp16
    out = out.transpose(-1, -2).to(feat.dtype).reshape(batch, ch, ht, wd)
    return x + self.proj(out)

Attention.forward = _flash_forward
print('Attention.forward patched → F.scaled_dot_product_attention (flash, O(N) memory)')

In [ ]:
@torch.no_grad()
def segment_volume_batched(model, img_ch, slices_t, H, W):
    """
    slices_t : (D, img_ch, IMG_SIZE, IMG_SIZE) float32 tensor on CPU in [-1, 1]
    Returns  : (D, H, W) uint8 numpy mask
    """
    D        = slices_t.shape[0]
    pred_vol = np.zeros((D, H, W), dtype=np.uint8)

    for b0 in range(0, D, INFER_BATCH):
        b1    = min(b0 + INFER_BATCH, D)
        img_b = slices_t[b0:b1].to(DEVICE)          # (B, img_ch, 256, 256)

        with torch.autocast('cuda', dtype=torch.float16,
                            enabled=(DEVICE.type == 'cuda')):
            pred = diffusion.ddim_sample(model, img_b, num_steps=DDIM_STEPS)
                                                     # (B, 1, 256, 256)

        # Resize back to original H × W
        pred_hw = F.interpolate(pred.float(), size=(H, W),
                                mode='bilinear', align_corners=False)
        pred_np = (pred_hw.squeeze(1).cpu().numpy() > 0.0).astype(np.uint8)
        pred_vol[b0:b1] = pred_np

    return pred_vol


def preprocess_volume(grey_arr, img_ch):
    """Build (D, img_ch, IMG_SIZE, IMG_SIZE) float32 CPU tensor for one volume."""
    D = grey_arr.shape[0]
    slices = []
    for sl in range(D):
        channels = [torch.from_numpy(_norm(grey_arr[sl])).unsqueeze(0)]
        if img_ch == 2:
            channels.append(torch.zeros_like(channels[0]))
        img_t = torch.cat(channels, dim=0) * 2.0 - 1.0   # (img_ch, H, W) in [-1,1]
        img_r = F.interpolate(
            img_t.unsqueeze(0),
            size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False,
        ).squeeze(0)                                       # (img_ch, 256, 256)
        slices.append(img_r)
    return torch.stack(slices)                             # (D, img_ch, 256, 256)


for dcm_path in dcm_files:
    idx      = re.search(r'Aug_(\d+)\.dcm', dcm_path).group(1)
    out_path = os.path.join(OUTPUT_DIR, f'Aug_{idx}_seg.npz')

    if os.path.exists(out_path):
        existing = set(np.load(out_path).files)
        if set(models.keys()).issubset(existing):
            print(f'Skipping (done): Aug_{idx}')
            continue

    print(f'\nProcessing: Aug_{idx}')
    ds       = pydicom.dcmread(dcm_path)
    grey_arr = ds.pixel_array.astype(np.float32)
    D, H, W  = grey_arr.shape
    print(f'  Shape: {grey_arr.shape}')

    all_masks = {}
    if os.path.exists(out_path):
        all_masks = dict(np.load(out_path))

    # Preprocess slices once — reused for all muscles
    max_ch = max(img_ch for _, img_ch in models.values())
    slices_t = preprocess_volume(grey_arr, max_ch)
    print(f'  Preprocessed {D} slices → {tuple(slices_t.shape)}')

    for muscle, (model, img_ch) in models.items():
        if muscle in all_masks:
            print(f'  {muscle}: already done')
            continue
        print(f'  {muscle} ... (INFER_BATCH={INFER_BATCH})', end=' ', flush=True)
        # Trim to the channel count this model expects
        pred = segment_volume_batched(model, img_ch, slices_t[:, :img_ch], H, W)
        all_masks[muscle] = pred
        print(f'{int(pred.sum()):,} voxels')

    np.savez_compressed(out_path, **all_masks)
    print(f'  Saved → {out_path}')

print('\nAll done.')

In [ ]:
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.npz')))
print(f'Output files: {len(results)} / {len(dcm_files)}')
if results:
    s = np.load(results[0])
    for k in sorted(s.files):
        print(f'  {k}: shape={s[k].shape}  voxels={int(s[k].sum()):,}')